In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [3]:
### Load the IMDB dataset word index

word_index = imdb.get_word_index()
reverse_word_index = {value: key for (key, value) in word_index.items()}

In [4]:
### LOad the pre-trained model with RELU activation function and dropout rate of 0.2

model=tf.keras.models.load_model('simplernn_imdb.keras')

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,052 (10.02 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,313,027 (5.01 MB)

In [6]:
model.get_weights()

[array([[ 0.43607363, -0.00536366,  0.03361686, ...,  0.07917333,
         -0.40461057,  0.00680549],
        [ 0.04038474,  0.01146797, -0.0025425 , ...,  0.03945018,
         -0.06925793, -0.02957668],
        [ 0.11385888,  0.05384539, -0.01869379, ...,  0.0131035 ,
          0.01629145,  0.00332238],
        ...,
        [-0.01997909,  0.12613288,  0.04646273, ..., -0.01134879,
          0.0607536 , -0.04430918],
        [ 0.02364576,  0.06996778, -0.06550489, ..., -0.04796188,
          0.00792988, -0.05592313],
        [-0.08844335, -0.07718361,  0.04248669, ...,  0.06585831,
          0.01260685,  0.00075061]], shape=(10000, 128), dtype=float32),
 array([[-0.03521894, -0.02180647,  0.10936337, ...,  0.1016002 ,
          0.08404616,  0.01262969],
        [ 0.05920905, -0.1068214 , -0.02431758, ...,  0.09862227,
         -0.01752452, -0.1049925 ],
        [-0.09662779, -0.13561174,  0.02662847, ...,  0.10271223,
         -0.0078395 , -0.02587946],
        ...,
        [-0.0460570

In [7]:
### Function to decode the review from integers to words

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

### Function to preprocess user input

def preprocess_text(text):
    # Tokenize the text into words
    words = text.lower().split()
    
    # Convert words to integers using the word index
    encoded_review = [word_index.get(word, 2) + 3 for word in words]  # 2 is for unknown words
    
    # Pad the sequence to a fixed length (e.g., 500)
    padded_review = pad_sequences([encoded_review], maxlen=500)
    
    return padded_review

In [9]:
### Prediction Function

def predict_sentiment(review):
    # Preprocess the review
    processed_review = preprocess_text(review)
    
    # Make prediction
    prediction = model.predict(processed_review)
    
    # Interpret the prediction
    sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
    
    return sentiment, prediction[0][0]

In [11]:
### Example review for testing

example_review = "The movie was fantastic! I really enjoyed it."
sentiment,score = predict_sentiment(example_review)
print(f"Review: {example_review}")
print(f"Predicted Sentiment: {sentiment} (Score: {score})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
Review: The movie was fantastic! I really enjoyed it.
Predicted Sentiment: Positive (Score: 0.8093366026878357)
